![Databricks Academy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/db-academy.png)

# 14 Bonus Lab - AUTO CDC INTO with SCD Type 1

##### NOTE: The AUTO CDC APIs replace the APPLY CHANGES APIs, and have the same syntax. The APPLY CHANGES APIs are still available, but Databricks recommends using the AUTO CDC APIs in their place.

### Estimated Duration: ~15-20 minutes

### Learning Objectives

By the end of this lesson, you will be able to:
- Use `AUTO CDC INTO` to perform Change Data Capture (CDC) using SCD Type 1.


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">This is an optional lab that can be completed after class if you're interested in practicing CDC</strong>
  <div style="color:#333;">

In this lab, you will use Change Data Capture (CDC) to detect changes and apply them using SCD Type 1 logic (overwrite, no historical records).

  </div>
</div>




## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this lab.

In [0]:
%run ./Includes/Classroom-Setup-Lab-cdc

## B. SCENARIO

Your data engineering team wants to build a Apache Spark™ Declarative Pipeline to maintain a record of current employees without keeping historical data (SCD Type 1).

The project has been started, but the final step of **updating the silver table with current employee records** has not yet been completed.

There are already two files in a cloud storage location that contain information about employees and employee updates.

### REQUIREMENTS:
It's your job to complete the Spark Declarative Pipeline by adding the `AUTO CDC INTO` statement to perform SCD Type 1.

Follow the steps below to complete your task.

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 22px 26px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/genie-code.png" style="height: 44px; margin-bottom: 10px;">

  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 12px;">
    Need Help? Use Genie Code
  </div>

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.6; margin-bottom: 16px;">
    Genie is an AI-powered assistant that can help you as you work through this lab. 
    Use it if you get stuck or want a little extra guidance.
  </div>

  <a href="https://docs.databricks.com/aws/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">AWS</a> |
  <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">Azure</a> |
  <a href="https://docs.databricks.com/gcp/en/genie-code/" target="_blank" style="display: inline-block; background: #1B5162; color: white; font-size: 14pt; font-weight: 700; padding: 10px 22px; border-radius: 8px; text-decoration: none;">GCP</a>

</div>

</div>

## C. Explore the Raw Data Source Files

1. Run the cell below to programmatically view the files in your `/Volumes/labuser/sdp_lab_1_bronze/lab_files` volume.

    Confirm that you see **employees_1.csv** and **employees_2.csv**.

**NOTE:** You can also manually navigate to your **labuser.sdp_lab_1_bronze.lab_files** volume and view the files in the volume.


In [0]:
%python
spark.sql(f'LIST "/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files"').display()

2. Query the 2 CSV files in that volume.

    Notice the following:

    - The files contain a list of employees.

    - The **employees_1.csv** contains the initial employees.

    - The **employees_2.csv** contains an update, a delete, and two new employees.

    - The **Operation** column provides information about the action for each record (new employee, update employee information, or delete employee).

    - The **ProcessDate** column indicates when the records were processed (acts as a sequence column).

    - In total, there are 10 rows.

        - There are two duplicate **EmployeeID** values:
          - **EmployeeID 1** – Sophia was an employee, then should be **deleted**.
          - **EmployeeID 3** – Liam received a bonus, and his **Salary** needs to be **updated**.

        - **Employee 6 & 7** - New employees from the **employees_2.csv** file.

In [0]:
SELECT
  _metadata.file_name as source_file,
  *
FROM read_files(
  '/Volumes/' || my_catalog || '/sdp_lab_1_bronze/lab_files',
  format => 'CSV'
)
ORDER BY source_file, EmployeeID, ProcessDate DESC;

3. Looking at the output from above, our final table after applying SCD Type 1 on the two files should:

   - Contain 6 rows of data:
      - remove the **EmployeeID** with a `null` value (removed with a data quality expectation)
      - delete **EmployeeID** 1 (employee who left)

   - **EmployeeID 3** should have a current salary of 100,000 and only one row of data.

   - **EmployeeID 6 & 7** are new employees from **employees_2.csv** file.

   - No historical data should be tracked.

<br></br>

**FINAL TABLE OUTPUT**
| EmployeeID | FirstName | Country | Department | Salary | HireDate   | ProcessDate |
|------------|-----------|---------|------------|--------|------------|-------------|
| 2          | Nikos     | GR      | IT         | 55000  | 2025-04-10 | 2025-06-05  |
| 3          | Liam      | US      | Sales      | **100000** | 2025-05-03 | **2025-06-22**  |
| 4          | Elena     | GR      | IT         | 53000  | 2025-06-04 | 2025-06-05  |
| 5          | James     | US      | IT         | 60000  | 2025-06-05 | 2025-06-05  |
| 6          | Emily     | US      | Enablement | 80000  | 2025-06-09 | **2025-06-22**  |
| 7          | Yannis    | GR      | HR         | 70000  | 2025-06-20 | **2025-06-22**  |


## D. TO DO: Complete the Pipeline with SCD Type 1

### D1. Create the Spark Declarative Pipeline

1. Run the cell below to create your starter Spark Declarative Pipeline for this lab. The pipeline will set the following for you:
    - Your default catalog: **labuser**
    - Your configuration parameter: `source` = `/Volumes/labuser/sdp_lab_1_bronze/lab_files`

    **NOTE:** If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

    To delete the pipeline:

    a. Select **Jobs and Pipelines** from the far-left navigation bar.

    b. Find the pipeline you want to delete.

    c. Click the three-dot menu ![ellipsis icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/ellipsis_icon.png).

    d. Select **Delete**.

**NOTE:**  The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'14 - CDC Lab Starter Project - {my_catalog}',
    root_path_folder_name='14 - CDC Lab Starter Project',
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['cdc_type_1_pipeline'],
    configuration={
        'source': f'/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files'
    }
)

2. Complete the following steps to open the starter Spark Declarative Pipeline project for this lab:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **14 - CDC Lab Starter Project - labuser** pipeline.

   c. In the **Pipeline details** pane on the far right, select **Open in Editor** (link to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab you should see the folder: **cdc_type_1_pipeline**.

   e. Open the **cdc_type_1_pipeline** folder and select the **cdc_employees.sql** file.

### D2. Complete the `cdc_employees.sql` File with `AUTO CDC INTO`

1. Review the code in the `cdc_employees.sql` file and complete the `AUTO CDC INTO` statement to perform SCD Type 1.
    - For simplicity in training, all code for the pipeline is in one file **cdc_employees.sql**.

    - Walk through the **cdc_employees.sql** file and read the comments.

    - The **bronze** and **silver** table code is completed for you. You just need to complete the `AUTO CDC INTO` statement.

    - If you need the solution for `AUTO CDC INTO`, expand the cell below.

  2. Complete the `cdc_employees.sql` file and run the pipeline.

AUTO CDC INTO (Apache Spark™ Declarative Pipelines):
[AWS](https://docs.databricks.com/aws/en/dlt-ref/dlt-sql-ref-apply-changes-into) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into) |
[GCP](https://docs.databricks.com/gcp/en/dlt-ref/dlt-sql-ref-apply-changes-into)

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
-- Create the empty streaming table
CREATE OR REFRESH STREAMING TABLE sdp_lab_2_silver.current_employees_silver_demo14;

-- Perform CDC SCD Type 1
CREATE FLOW scd_type_1_flow AS
AUTO CDC INTO sdp_lab_2_silver.current_employees_silver_demo14  -- Target table to update with SCD Type 1 (or 2)
FROM STREAM sdp_lab_1_bronze.employees_bronze_clean_demo14      -- Source streaming table
KEYS (EmployeeID)
APPLY AS DELETE WHEN Operation = 'delete'
SEQUENCE BY ProcessDate
COLUMNS * EXCEPT (Operation)
STORED AS SCD TYPE 1;
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

## E. Explore Your CDC SCD Type 1 Streaming Table

After you have completed the `AUTO CDC INTO` statement in the **cdc_employees.sql** file, compare your results to the solution image below.

**FINAL PIPELINE RUN**

![Lab 7 Pipeline Run](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lab-2/lab_7_pipelinerun.png)

1. Run the cell below to view the data in your **sdp_lab_2_silver.current_employees_silver_demo14** streaming table that applied SCD Type 1, and compare it to the solution below.

    Notice that with SCD Type 1, no historical data is kept.

**FINAL TABLE SOLUTION**
| EmployeeID | FirstName | Country | Department | Salary | HireDate   | ProcessDate |
|------------|-----------|---------|------------|--------|------------|-------------|
| 2          | Nikos     | GR      | IT         | 55000  | 2025-04-10 | 2025-06-05  |
| 3          | Liam      | US      | Sales      | **100000** | 2025-05-03 | **2025-06-22**  |
| 4          | Elena     | GR      | IT         | 53000  | 2025-06-04 | 2025-06-05  |
| 5          | James     | US      | IT         | 60000  | 2025-06-05 | 2025-06-05  |
| 6          | Emily     | US      | Enablement | 80000  | 2025-06-09 | **2025-06-22**  |
| 7          | Yannis    | GR      | HR         | 70000  | 2025-06-20 | **2025-06-22**  |

In [0]:
SELECT *
FROM sdp_lab_2_silver.current_employees_silver_demo14
ORDER BY EmployeeID

## F. CHALLENGE SCENARIO
### Duration: ~10 minutes

**NOTE:** *If you finish early in a live class, feel free to complete the challenge below. The challenge is optional and most likely won't be completed during the live class. Only continue if your Spark Declarative Pipeline was set up correctly in the previous section by comparing your pipeline to the solution image.*

**SCENARIO:** In the challenge, you will land a new CSV file in your **lab_files** cloud storage volume and rerun the pipeline to watch the Spark Declarative Pipeline perform CDC SCD Type 1 on the new data.

1. Run the cell below to land another file in your **lab_files** cloud storage location and confirm that 3 CSV files exist.

In [0]:
%python

## Find data in workspace data folder
data_path = "/Volumes/dbacademy/default/data"

## Land JSON files to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/lab_files',
    target_volume_path=f'/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files',
    n=3
)


spark.sql(f"LIST '/Volumes/{my_catalog}/sdp_lab_1_bronze/lab_files'").display()

2. Query the **employees_3.csv** file. Notice the following:

   - **EmployeeID** values **2** and **6** need to be removed.

   - **EmployeeID 8** is a new employee in our company.


In [0]:
SELECT
  _metadata.file_name as source_file,
  *
FROM read_files(
  '/Volumes/' || my_catalog || '/sdp_lab_1_bronze/lab_files/employees_3.csv',
  format => 'CSV'
);

3. Go back to your pipeline and select **Run pipeline**. Examine the pipeline run. Confirm it shows the following:

![Lab 7 Challenge Run](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lab-2/lab_7_challengesolution.png)

4. Run the cell below to query the table **sdp_lab_2_silver.current_employees_silver_demo14** and view the results. Notice that:

   - The two employees (**EmployeeID** 2 and 6) were deleted.

   - **EmployeeID 8** was added.

   - No historical data is kept with SCD Type 1.

    **NOTE:** If you ran the solution pipeline, the streaming table is named **sdp_lab_2_silver.current_employees_silver_demo14_solution**.


    **FINAL TABLE**
| EmployeeID | FirstName  | Country | Department | Salary  | HireDate   | ProcessDate |
|------------|------------|---------|------------|---------|------------|-------------|
| 3          | Liam       | US      | Sales      | 100000  | 2025-05-03 | 2025-06-22  |
| 4          | Elena      | GR      | IT         | 53000   | 2025-06-04 | 2025-06-05  |
| 5          | James      | US      | IT         | 60000   | 2025-06-05 | 2025-06-05  |
| 7          | Yannis     | GR      | HR         | 70000   | 2025-06-20 | 2025-06-22  |
| 8          | Panagiotis | GR      | Enablement | 90000   | 2025-07-01 | 2025-07-22  |

In [0]:
SELECT *
FROM sdp_lab_2_silver.current_employees_silver_demo14
ORDER BY EmployeeID;


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
